In [1]:
# Uncomment in new environment.
#!pip install -r ../requirements.txt
#!git clone https://github.com/mohamesb/PyKalmanSmoothingGlucose.git

## 30 Minutes - Prediction Horizon

### Data Processing

In [2]:
# Cell 1: Processing data + feature engineering & selection
# Produces: per_subject_train, per_subject_test, cg_scaler, feat_scaler, metadata in memory
# NOTE: run this cell first.

# Install smoothing helper from the repo you referenced
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/miriamkw/PyKalmanSmoothingGlucose.git"], stdout=subprocess.DEVNULL)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from smoother.smooth_SMBG_data import smooth_smbg_data   # from repo
from datetime import timedelta

# Parameters (easy to change)
DATA_PATH = "../data/OhioT1DM.csv"
N_IN = 12   # input length (12 * 5min = 60 min history) - configurable
N_OUT = 6   # 6 steps * 5min = 30 min horizon
RANDOM_STATE = 42

# ------------- Processing data -------------
df = pd.read_csv(DATA_PATH)
# Parse datetime and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['id', 'date']).reset_index(drop=True)

# We'll keep a set of features to use
features = ["CGM","carbs","bolus","basal","galvanic_skin_response","skin_temp",
            "acceleration","workout_intensity","workout_duration","heartrate","air_temp","steps"]

# Per-subject smoothing + forward fill using smoother.smooth_SMBG_data
per_subject = {}
for sid, g in df.groupby('id'):
    g = g.sort_values('date').reset_index(drop=True)
    t = g['date'].values
    y = g['CGM'].values
    # use smoother; the function returns dict with 'y_smoothed' and 'y_smoothed_at_tout'
    try:
        res = smooth_smbg_data(t, y, outlier_removal=1, dynamic_model=2)
        y_smooth = res.get('y_smoothed_at_tout', res.get('y_smoothed', None))
        if y_smooth is None:
            y_smooth = np.array(y)
    except Exception as e:
        # fallback: simple forward-fill/linear interpolation if smoother fails
        ys = pd.Series(y).interpolate(limit_direction='both').fillna(method='ffill').fillna(method='bfill').values
        y_smooth = ys
    g2 = g.copy()
    g2['CGM_smoothed'] = y_smooth
    # forward-fill other signals and interpolate small gaps
    for col in features:
        if col not in g2.columns:
            g2[col] = 0.0
    g2[features] = g2[features].interpolate(limit_direction='both').fillna(method='ffill').fillna(0.0)
    per_subject[sid] = g2.reset_index(drop=True)

# ------------- Train/test split by subject (no leakage) -------------
subject_ids = sorted(list(per_subject.keys()))
train_ids, test_ids = train_test_split(subject_ids, test_size=0.2, random_state=RANDOM_STATE)
print(f"Subjects total: {len(subject_ids)}  train: {len(train_ids)}  test: {len(test_ids)}")

# ------------- Feature scaling (global scalers) -------------
# For CGM use MinMaxScaler to keep outputs in sensible range for NN; store scaler for inverse later
all_cgm = np.concatenate([per_subject[s]['CGM_smoothed'].values for s in train_ids])
cg_scaler = MinMaxScaler(feature_range=(0,1))
cg_scaler.fit(all_cgm.reshape(-1,1))

# For other features use a standard scaler (fit on train subjects)
all_feats = []
for sid in train_ids:
    df_s = per_subject[sid]
    # combine non-CGM features
    feats = df_s[['carbs','bolus','basal','galvanic_skin_response','skin_temp',
                  'acceleration','workout_intensity','workout_duration','heartrate','air_temp','steps']].values
    all_feats.append(feats)
all_feats = np.vstack(all_feats)
feat_scaler = StandardScaler()
feat_scaler.fit(all_feats)

# ------------- Sequence creation helpers -------------
def create_sequences(df_s, n_in=N_IN, n_out=N_OUT):
    """
    For a subject dataframe, produce sequences X (n_samples, n_in, n_features) and y (n_samples, n_out, 1)
    X includes CGM (scaled) + other scaled features. y is CGM (scaled).
    """
    X_list = []
    y_list = []
    times = []
    cgm = df_s['CGM_smoothed'].values
    other = df_s[['carbs','bolus','basal','galvanic_skin_response','skin_temp',
                  'acceleration','workout_intensity','workout_duration','heartrate','air_temp','steps']].values
    # scale
    cgm_s = cg_scaler.transform(cgm.reshape(-1,1)).reshape(-1)
    other_s = feat_scaler.transform(other)
    L = len(df_s)
    for i in range(n_in, L - n_out + 1):
        xin = np.column_stack([cgm_s[i-n_in:i], other_s[i-n_in:i]])
        yout = cgm_s[i:i+n_out].reshape(n_out,1)  # next n_out steps
        X_list.append(xin)
        y_list.append(yout)
        times.append(df_s['date'].iloc[i:i+n_out].values)
    if len(X_list) == 0:
        return np.zeros((0,n_in,cgm_s.shape[0]+other_s.shape[1])), np.zeros((0,n_out,1)), []
    X = np.stack(X_list)
    y = np.stack(y_list)
    return X, y, times

# ------------- Build per_subject_train and per_subject_test -------------
per_subject_train = {}
per_subject_test = {}
for sid in train_ids:
    X,y,times = create_sequences(per_subject[sid])
    per_subject_train[sid] = {'X':X, 'y':y, 'times':times}
for sid in test_ids:
    X,y,times = create_sequences(per_subject[sid])
    per_subject_test[sid] = {'X':X, 'y':y, 'times':times}

# Convenience aggregated arrays for NN training (stack across train subjects)
X_tr_all = np.concatenate([per_subject_train[s]['X'] for s in per_subject_train if per_subject_train[s]['X'].size], axis=0) if any([per_subject_train[s]['X'].size for s in per_subject_train]) else np.zeros((0,N_IN,1+11))
y_tr_all = np.concatenate([per_subject_train[s]['y'] for s in per_subject_train if per_subject_train[s]['y'].size], axis=0) if any([per_subject_train[s]['y'].size for s in per_subject_train]) else np.zeros((0,N_OUT,1))

print("Shapes: X_train_all", X_tr_all.shape, "y_train_all", y_tr_all.shape)
# keep these in notebook memory for later model cells


Autodetected mg/dL as unit
Smoother flagged measurement 1743 as outlier: t = 10045.0, y = 8.769008769008769 [mmol/L].
Smoother flagged measurement 8689 as outlier: t = 47370.0, y = 6.382506382506382 [mmol/L].
Smoother flagged measurement 14727 as outlier: t = 80995.0, y = 16.87201687201687 [mmol/L].
Smoother flagged measurement 14728 as outlier: t = 81000.0, y = 12.654012654012654 [mmol/L].
Smoother needs a second pass due to outliers detected. Total # outliers in input data: 4


C:\Users\msbdj\AppData\Local\Temp\ipykernel_13416\5754523.py:55: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g2[features] = g2[features].interpolate(limit_direction='both').fillna(method='ffill').fillna(0.0)


Autodetected mg/dL as unit
Smoother flagged measurement 975 as outlier: t = 5055.0, y = 4.828504828504828 [mmol/L].
Smoother flagged measurement 1563 as outlier: t = 7995.0, y = 15.54001554001554 [mmol/L].
Smoother flagged measurement 1564 as outlier: t = 8000.0, y = 14.263514263514264 [mmol/L].
Smoother flagged measurement 1566 as outlier: t = 8010.0, y = 9.268509268509268 [mmol/L].
Smoother flagged measurement 1574 as outlier: t = 8050.0, y = 13.32001332001332 [mmol/L].
Smoother flagged measurement 1575 as outlier: t = 8055.0, y = 12.543012543012543 [mmol/L].
Smoother flagged measurement 1577 as outlier: t = 8065.0, y = 8.935508935508935 [mmol/L].
Smoother flagged measurement 1608 as outlier: t = 8220.0, y = 9.213009213009213 [mmol/L].
Smoother flagged measurement 1610 as outlier: t = 8230.0, y = 6.715506715506716 [mmol/L].
Smoother flagged measurement 1615 as outlier: t = 8255.0, y = 9.046509046509046 [mmol/L].
Smoother flagged measurement 1617 as outlier: t = 8265.0, y = 5.99400599

C:\Users\msbdj\AppData\Local\Temp\ipykernel_13416\5754523.py:55: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g2[features] = g2[features].interpolate(limit_direction='both').fillna(method='ffill').fillna(0.0)


Autodetected mg/dL as unit
Smoother flagged measurement 2427 as outlier: t = 13630.0, y = 8.658008658008658 [mmol/L].
Smoother flagged measurement 4178 as outlier: t = 24540.0, y = 9.768009768009767 [mmol/L].
Smoother flagged measurement 4179 as outlier: t = 24545.0, y = 13.375513375513375 [mmol/L].
Smoother flagged measurement 5050 as outlier: t = 29290.0, y = 11.211011211011211 [mmol/L].
Smoother flagged measurement 5051 as outlier: t = 29295.0, y = 7.7145077145077146 [mmol/L].
Smoother flagged measurement 7860 as outlier: t = 48095.0, y = 9.712509712509712 [mmol/L].
Smoother flagged measurement 11093 as outlier: t = 74020.0, y = 3.94050394050394 [mmol/L].
Smoother flagged measurement 11094 as outlier: t = 74025.0, y = 5.55000555000555 [mmol/L].
Smoother needs a second pass due to outliers detected. Total # outliers in input data: 8


...

Autodetected mg/dL as unit
Smoother flagged measurement 267 as outlier: t = 2360.0, y = 16.206016206016205 [mmol/L].
Smoother flagged measurement 4922 as outlier: t = 26710.0, y = 9.879009879009878 [mmol/L].
Smoother flagged measurement 7918 as outlier: t = 46800.0, y = 5.328005328005328 [mmol/L].
Smoother flagged measurement 7997 as outlier: t = 47445.0, y = 5.55000555000555 [mmol/L].
Smoother flagged measurement 8020 as outlier: t = 47565.0, y = 9.324009324009324 [mmol/L].
Smoother flagged measurement 8183 as outlier: t = 48895.0, y = 7.548007548007548 [mmol/L].
Smoother flagged measurement 8878 as outlier: t = 54555.0, y = 11.71051171051171 [mmol/L].
Smoother flagged measurement 9048 as outlier: t = 55425.0, y = 8.935508935508935 [mmol/L].
Smoother flagged measurement 9052 as outlier: t = 55445.0, y = 6.271506271506271 [mmol/L].
Smoother flagged measurement 9054 as outlier: t = 55455.0, y = 8.824508824508824 [mmol/L].
Smoother flagged measurement 9055 as outlier: t = 55460.0, y = 8.

C:\Users\msbdj\AppData\Local\Temp\ipykernel_13416\5754523.py:55: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g2[features] = g2[features].interpolate(limit_direction='both').fillna(method='ffill').fillna(0.0)


Autodetected mg/dL as unit
Smoother flagged measurement 624 as outlier: t = 8070.0, y = 3.4410034410034407 [mmol/L].
Smoother flagged measurement 625 as outlier: t = 8075.0, y = 6.271506271506271 [mmol/L].
Smoother flagged measurement 779 as outlier: t = 9595.0, y = 7.104007104007104 [mmol/L].
Smoother flagged measurement 1734 as outlier: t = 14395.0, y = 13.264513264513264 [mmol/L].
Smoother flagged measurement 2199 as outlier: t = 19905.0, y = 6.438006438006438 [mmol/L].
Smoother flagged measurement 2200 as outlier: t = 19910.0, y = 9.37950937950938 [mmol/L].
Smoother flagged measurement 2367 as outlier: t = 20930.0, y = 10.71151071151071 [mmol/L].
Smoother flagged measurement 2508 as outlier: t = 21635.0, y = 10.989010989010989 [mmol/L].
Smoother flagged measurement 2730 as outlier: t = 22750.0, y = 5.772005772005771 [mmol/L].
Smoother flagged measurement 3916 as outlier: t = 29285.0, y = 5.772005772005771 [mmol/L].
Smoother flagged measurement 5538 as outlier: t = 38705.0, y = 7.10

C:\Users\msbdj\AppData\Local\Temp\ipykernel_13416\5754523.py:55: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  g2[features] = g2[features].interpolate(limit_direction='both').fillna(method='ffill').fillna(0.0)


Subjects total: 12  train: 9  test: 3
Shapes: X_train_all (142531, 12, 12) y_train_all (142531, 6, 1)


## Zero Order Model - Persistance:

In [3]:
# Cell 3: Zero-order (persistence) baseline
# Structure: Processing data, Feature engineering & selection, Training model, Test model
# This is trivial — no training. For multi-step we repeat last observed CGM value across all future steps.

import numpy as np

def persistence_predict_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, N_OUT, 1))
    X = info['X']  # shape (n_samples, n_in, n_feats)
    # the first channel was scaled CGM at each timestep
    last_cgm_scaled = X[:, -1, 0]  # (n_samples,)
    # For each horizon step, predict the previous cgm (repeat); shape (n_samples, n_out,1)
    preds = np.tile(last_cgm_scaled.reshape(-1,1,1), (1, N_OUT, 1))
    return preds

persistence_model = {"predict": persistence_predict_subject}


## Pure LSTM Model - Vanilla LSTM:

In [4]:
# Cell 2: Vanilla LSTM (multi-step multi-input)
# Headings: Processing data, Feature engineering & selection, Training model, Test model
# Expects variables from Cell 1: per_subject_train, per_subject_test, X_tr_all, y_tr_all, cg_scaler

import numpy as np
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, TimeDistributed
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# ---------- Processing data (use aggregated arrays from preproc cell) ----------
X_train = X_tr_all  # shape (N_samples, n_in, n_features)
y_train = y_tr_all  # shape (N_samples, n_out, 1)
batch_size = 64
epochs = 20

n_in = X_train.shape[1]
n_feats = X_train.shape[2]
n_out = y_train.shape[1]

# ---------- Feature engineering: no further features added; keep shape as is ----------
# ---------- Custom surrogate loss (differentiable approximation of compound metric) ----------
# We use RMSE + alpha * soft-glycemia mismatch (soft classes via logistic around thresholds)
alpha = 0.6
k_soft = 0.08  # slope for soft thresholds (tuneable)

def soft_glycemia_probs(y):
    # y: tensor of glucose values in scaled space -> convert to mg/dL by inverse scaling inside loss
    y = tf.reshape(y, (-1,))  # flatten
    # inverse scale using stored scaler parameters (min/max). We compute mg/dL approximations:
    min_v = float(cg_scaler.data_min_[0]); max_v = float(cg_scaler.data_max_[0])
    y_mgdl = y * (max_v - min_v) + min_v
    # probabilities: hypo prob ~ sigmoid(k*(70 - y)), hyper prob ~ sigmoid(k*(y - 180))
    hypo = tf.sigmoid(k_soft * (70.0 - y_mgdl))
    hyper = tf.sigmoid(k_soft * (y_mgdl - 180.0))
    norm = 1.0 - tf.clip_by_value(hypo + hyper, 0.0, 1.0)
    probs = tf.stack([hypo, norm, hyper], axis=1)
    return probs

def lstm_surrogate_loss(y_true, y_pred):
    # y_true/y_pred shapes (batch, n_out, 1)
    # RMSE term
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    rmse = tf.sqrt(mse + 1e-8)
    # soft glycemia mismatch: compute probs and L1 difference
    pt = soft_glycemia_probs(y_true)
    pp = soft_glycemia_probs(y_pred)
    soft_diff = tf.reduce_mean(tf.abs(pt - pp))
    return rmse + alpha * soft_diff

# ---------- Training model ----------
tf.keras.backend.clear_session()
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(n_in, n_feats)),
    LSTM(32, return_sequences=False),
    Dense(n_out*1),
    # reshape at training-time to (n_out,1) in loss; Keras expects final shape (batch, n_out)
])
# we'll wrap outputs to (batch, n_out, 1) by custom lambda at training time in fit callbacks,
# but simplest is to compile with target flattened to (n_out,) so cast both
model.add(Dense(n_out, activation='linear'))

model.compile(optimizer=Adam(learning_rate=1e-3), loss=lstm_surrogate_loss)
print(model.summary())

# prepare training targets flattened to (batch, n_out)
y_train_flat = y_train.reshape((y_train.shape[0], n_out))

model.fit(X_train, y_train_flat, epochs=epochs, batch_size=batch_size, verbose=2)

# ---------- Test model: prediction helper ----------
def lstm_predict_on_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, n_out,1))
    X = info['X']
    y_hat_flat = model.predict(X, batch_size=64)
    y_hat = y_hat_flat.reshape((-1, n_out,1))
    return y_hat

# expose model variable for later evaluation
lstm_model = model


c:\Users\msbdj\Documents\Thesis\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 12, 64)         │        19,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           198 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │            42 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,368 (126.44 KB)

 Trainable params: 32,368 (126.44 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
2228/2228 - 26s - 12ms/step - loss: nan
Epoch 2/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 3/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 4/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 5/20
2228/2228 - 23s - 10ms/step - loss: nan
Epoch 6/20
2228/2228 - 40s - 18ms/step - loss: nan
Epoch 7/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 8/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 9/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 10/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 11/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 12/20
2228/2228 - 23s - 10ms/step - loss: nan
Epoch 13/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 14/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 15/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 16/20
2228/2228 - 21s - 10ms/step - loss: nan
Epoch 17/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 18/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 19/20
2228/2228 - 22s - 10ms/step - loss: nan
Epoch 20/20
2228

## Bergman Minimal Model:

In [6]:
# Corrected Cell 4: Simple Bergman minimal-like model (with cleaning & diagnostics)
import numpy as np
from scipy.optimize import least_squares

dt_min = 5.0
dt = dt_min / 60.0

# Build training dataset (as before)
train_sequences = []
for sid, info in per_subject_train.items():
    subj_df = per_subject[sid]
    L = subj_df.shape[0]
    for i in range(N_IN, L - N_OUT + 1):
        G_init = subj_df['CGM_smoothed'].values[i-1]
        bolus_seq = subj_df['bolus'].values[i:i+N_OUT]
        carbs_seq = subj_df['carbs'].values[i:i+N_OUT]
        y_true = subj_df['CGM_smoothed'].values[i:i+N_OUT]
        train_sequences.append((G_init, bolus_seq, carbs_seq, y_true))

if len(train_sequences) == 0:
    raise RuntimeError("No training sequences found for Bergman fit. Check preprocessing cell.")

# --- CLEAN: remove any sequences with non-finite values ---
clean_sequences = []
bad_count = 0
for seq in train_sequences:
    G_init, bolus_seq, carbs_seq, y_true = seq
    ok = True
    # check all arrays/scalars for finiteness
    try:
        if not np.isfinite(float(G_init)):
            ok = False
    except Exception:
        ok = False
    for arr in (bolus_seq, carbs_seq, y_true):
        a = np.asarray(arr, dtype=float)
        if not np.all(np.isfinite(a)):
            ok = False
            break
    if ok:
        clean_sequences.append((float(G_init),
                                np.asarray(bolus_seq, dtype=float),
                                np.asarray(carbs_seq, dtype=float),
                                np.asarray(y_true, dtype=float)))
    else:
        bad_count += 1

print(f"Total sequences collected: {len(train_sequences)}  Clean: {len(clean_sequences)}  Removed (bad): {bad_count}")

if len(clean_sequences) == 0:
    raise RuntimeError("All sequences removed during cleaning — inspect raw data for NaNs/infs.")

# Model simulator (same as before, but ensure numeric types)
def simulate_bergman(params, seq):
    p1, p2, p3, k_i, Gb = params
    G_init, bolus_seq, carbs_seq, _ = seq
    N = len(bolus_seq)
    G = float(G_init)
    I = 0.0
    preds = np.empty(N, dtype=float)
    # guard small / large values of k_i for numeric stability
    k_i = float(k_i)
    for t in range(N):
        bt = float(bolus_seq[t])
        ct = float(carbs_seq[t])
        # insulin action update
        I = I * np.exp(-k_i) + bt
        meal_effect = p3 * ct
        dG = -p1 * (G - Gb) - p2 * I + meal_effect
        G = G + dG * (dt_min / 5.0)
        # Clip to physiologically plausible range to avoid any exploding numbers
        if not np.isfinite(G):
            # return an array of NaNs so residuals become non-finite (but we filtered input sequences already)
            return np.full(N, np.nan, dtype=float)
        preds[t] = G
    return preds

# Objective using cleaned sequences
def residuals_for_all(params):
    res_list = []
    for seq in clean_sequences:
        y_true = seq[3]
        y_pred = simulate_bergman(params, seq)
        # if simulation returned non-finite values, raise early with info
        if not np.all(np.isfinite(y_pred)):
            # return a vector containing a large finite value (least_squares expects finite), or raise
            # we choose to raise to get diagnostics
            raise RuntimeError("Non-finite predictions produced by simulate_bergman at params: " + str(params))
        res_list.append(y_pred - y_true)
    if len(res_list) == 0:
        return np.array([0.0])
    return np.concatenate(res_list)

# initial guess & bounds
p0 = np.array([0.01, 0.001, 0.01, 0.5, 100.0])
lb = [1e-5, 1e-6, 0.0, 1e-3, 40.0]
ub = [1.0, 1.0, 1.0, 5.0, 200.0]

print("Fitting bergman minimal model to training data (least squares)...")
try:
    res = least_squares(residuals_for_all, p0, bounds=(lb, ub), verbose=2, xtol=1e-4, max_nfev=500)
    p_opt = res.x
    print("Optimized params:", p_opt)
except Exception as e:
    # helpful debug info
    print("Error during least_squares:", repr(e))
    # try to evaluate residuals at p0 to see where non-finite appears
    try:
        r0 = residuals_for_all(p0)
        print("Residuals at p0 finite (unexpected).")
    except Exception as e2:
        print("Residuals at p0 raised:", repr(e2))
    raise

# Test-time predictor using fitted params (unchanged)
def bergman_predict_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    subj = per_subject[sid]
    preds_all = []
    L = subj.shape[0]
    for i in range(N_IN, L - N_OUT + 1):
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        y_pred = simulate_bergman(p_opt, (G_init, bolus_seq, carbs_seq, None))
        # if simulation produced non-finite (shouldn't happen), clip / replace
        if not np.all(np.isfinite(y_pred)):
            y_pred = np.nan_to_num(y_pred, nan=np.nanmedian(y_pred[np.isfinite(y_pred)]) if np.any(np.isfinite(y_pred)) else 100.0)
        y_pred_scaled = cg_scaler.transform(y_pred.reshape(-1,1)).reshape(-1,1)
        preds_all.append(y_pred_scaled)
    if len(preds_all) == 0:
        return np.zeros((0, N_OUT, 1))
    return np.stack(preds_all)

bergman_model = {"params": p_opt, "predict": bergman_predict_subject}


Total sequences collected: 142531  Clean: 141106  Removed (bad): 1425
Fitting bergman minimal model to training data (least squares)...
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         9.9702e+07                                    7.14e+06    
       1              2         9.7905e+07      1.80e+06       3.24e+01       2.87e+06    
       2              3         9.7523e+07      3.82e+05       2.26e+01       2.47e+07    
       3              4         9.7489e+07      3.39e+04       6.37e+00       6.41e+05    
       4              5         9.7489e+07      2.70e+02       5.30e-01       4.71e+03    
       5              6         9.7489e+07      8.90e+00       3.66e-02       2.17e+02    
       6              7         9.7489e+07      1.28e-01       3.77e-03       2.71e+01    
Both `ftol` and `xtol` termination conditions are satisfied.
Function evaluations 7, initial cost 9.9702e+07, final cost 9.7489e+07, fir

## Hybrid Model - ML Corrector:

In [7]:
# Cell 5: Hybrid ML-corrector (LSTM corrector on residuals)
# Processing data, Feature engineering & selection, Training model, Test model

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# ---------- Processing: build training set where target = residual (true - bergman_pred) ----------
X_corr = []
y_corr = []
for sid, info in per_subject_train.items():
    Xs = info['X']   # scaled inputs
    if Xs.shape[0] == 0:
        continue
    # compute bergman predictions for this subject's windows using bergman_model
    # we'll get scaled bergman preds per-sample by re-running bergman on internal data alignment
    subj = per_subject[sid]
    L = subj.shape[0]
    for i in range(N_IN, L - N_OUT + 1):
        xin = Xs[i - N_IN]  # input sequence aligned with create_sequences indexing
        # raw seq needed
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1,1)
        y_true_scaled = cg_scaler.transform(subj['CGM_smoothed'].values[i:i+N_OUT].reshape(-1,1)).reshape(-1,1)
        resid = (y_true_scaled - berg_pred_scaled).reshape(N_OUT)  # flatten n_out
        X_corr.append(xin)
        y_corr.append(resid)
if len(X_corr)==0:
    raise RuntimeError("No training data found for ML-corrector.")
X_corr = np.stack(X_corr)  # (N_samples, n_in, n_feats)
y_corr = np.stack(y_corr)  # (N_samples, n_out)

# ---------- Feature engineering: keep as-is ----------
n_in = X_corr.shape[1]
n_feats = X_corr.shape[2]
n_out = y_corr.shape[1]

# ---------- Surrogate loss same as LSTM (we use RMSE + soft glycemia) ----------
# re-use lstm_surrogate_loss defined earlier by adapting for shape (batch, n_out)
# We'll re-create a simple callable loss for compile (since earlier was bound to previous graph)
import tensorflow as tf
alpha = 0.6
k_soft = 0.08
min_v = float(cg_scaler.data_min_[0]); max_v = float(cg_scaler.data_max_[0])

def soft_glycemia_probs_flat(x_flat):
    # x_flat is (batch, n_out) scaled in [0,1]
    x = tf.reshape(x_flat, (-1,))
    x_mgdl = x * (max_v - min_v) + min_v
    hypo = tf.sigmoid(k_soft * (70.0 - x_mgdl))
    hyper = tf.sigmoid(k_soft * (x_mgdl - 180.0))
    norm = 1.0 - tf.clip_by_value(hypo + hyper, 0.0, 1.0)
    probs = tf.stack([hypo, norm, hyper], axis=1)
    return probs

def corrector_loss(y_true, y_pred):
    # y_true/y_pred shape (batch, n_out)
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    rmse = tf.sqrt(mse + 1e-8)
    # convert residual + bergman_pred to full glucose for class probs
    # We cannot access bergman_pred here; instead we compute soft penalty on the corrected final glucose by
    # assuming that later in the pipeline corrected_pred_full = bergman_pred_scaled + y_pred.
    # During training we can approximate the full by using bergman estimates computed offline:
    # For simplicity we apply soft class penalty only on y_true (resid->y_true+bergman) via a proxy:
    soft_diff = 0.0  # keep simple; main objective is RMSE of residuals
    return rmse + 0.1 * soft_diff

# ---------- Training model ----------
from tensorflow.keras import backend as K
tf.keras.backend.clear_session()
model_corr = Sequential([
    LSTM(32, return_sequences=False, input_shape=(n_in, n_feats)),
    Dense(n_out, activation='linear')
])
model_corr.compile(optimizer=Adam(1e-3), loss=corrector_loss)
print(model_corr.summary())
model_corr.fit(X_corr, y_corr, epochs=12, batch_size=64, verbose=2)

# ---------- Test model: produce corrected predictions per subject ----------
def ml_corrector_predict_subject(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0]==0:
        return np.zeros((0, N_OUT, 1))
    subj = per_subject[sid]
    L = subj.shape[0]
    preds_all = []
    for i in range(N_IN, L - N_OUT + 1):
        xin = info['X'][i - N_IN]  # aligned input
        xin_batch = xin.reshape(1, n_in, n_feats)
        resid_pred = model_corr.predict(xin_batch)[0]  # shape (n_out,)
        # bergman pred for this window
        G_init = subj['CGM_smoothed'].values[i-1]
        bolus_seq = subj['bolus'].values[i:i+N_OUT]
        carbs_seq = subj['carbs'].values[i:i+N_OUT]
        berg_pred = simulate_bergman(bergman_model['params'], (G_init, bolus_seq, carbs_seq, None))
        berg_pred_scaled = cg_scaler.transform(berg_pred.reshape(-1,1)).reshape(-1)
        corrected = berg_pred_scaled + resid_pred
        preds_all.append(corrected.reshape(-1,1))
    if len(preds_all) == 0:
        return np.zeros((0,N_OUT,1))
    return np.stack(preds_all)

ml_corrector_model = {"lstm_corrector": model_corr, "predict": ml_corrector_predict_subject}


c:\Users\msbdj\Documents\Thesis\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,958 (23.27 KB)

 Trainable params: 5,958 (23.27 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/12
2228/2228 - 15s - 7ms/step - loss: nan
Epoch 2/12
2228/2228 - 12s - 6ms/step - loss: nan
Epoch 3/12
2228/2228 - 12s - 6ms/step - loss: nan
Epoch 4/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 5/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 6/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 7/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 8/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 9/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 10/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 11/12
2228/2228 - 12s - 5ms/step - loss: nan
Epoch 12/12
2228/2228 - 12s - 5ms/step - loss: nan


## Evaluation

In [8]:
# Cell 6: Compound glucose prediction metric functions (own cell)
import numpy as np
from scipy.signal import correlate
from sklearn.metrics import recall_score

def rmse(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.nanmean(np.square(y_true - y_pred)))

def temporal_gain(y_true, y_pred, prediction_horizon=30):
    # returns lag in minutes (approx) where cross-correlation peaks, limited to [0, prediction_horizon]
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()
    cross_corr = correlate(y_true - np.nanmean(y_true), y_pred - np.nanmean(y_pred), mode='full')
    full_lags = np.arange(-len(y_true) + 1, len(y_pred))
    lag_min = 0
    lag_max = prediction_horizon // 5
    valid_mask = (full_lags >= lag_min) & (full_lags <= lag_max)
    if not np.any(valid_mask):
        return prediction_horizon
    valid_lags = full_lags[valid_mask]
    valid_cross_corr = cross_corr[valid_mask]
    if valid_cross_corr.size == 0:
        return prediction_horizon
    max_corr_idx = np.argmax(np.abs(valid_cross_corr))
    lag_steps = valid_lags[max_corr_idx]
    lag_minutes = int(lag_steps * 5)
    return lag_minutes

def g_mean(y_true, y_pred):
    # compute geometric mean of recalls across hypo/normo/hyper classes
    hypo_threshold = 70.0
    hyper_threshold = 180.0
    def map_cls(arr):
        arr = np.asarray(arr)
        labels = np.full(arr.shape, 1, dtype=int)
        labels[arr < hypo_threshold] = 0
        labels[arr > hyper_threshold] = 2
        return labels.flatten()
    true_labels = map_cls(y_true)
    pred_labels = map_cls(y_pred)
    # handle case where a class has zero support: recall_score throws error; we clip by small epsilon
    recalls = recall_score(true_labels, pred_labels, average=None, zero_division=0)
    # replace zeros with small epsilon to avoid geometric mean zero
    recalls = np.where(recalls == 0, 1e-10, recalls)
    gmean = np.exp(np.mean(np.log(recalls)))
    return gmean

def scale_error(metric_results, use_mg_dl=False):
    lower_bound = 1
    if use_mg_dl:
        lower_bound = 18
    max_abs = max(np.max(np.abs(metric_results)), lower_bound)
    return np.array([np.abs(val) / max_abs for val in metric_results])

def compound_glucose_prediction_metric(rmse_list, tg_list, g_mean_list, me_list=None, prediction_horizon=30, use_mg_dl=False):
    # Simple composite scoring:
    # scaled_rmse + scaled_temporal_gain + scaled_gmean_penalty
    scaled_rmse = scale_error(np.array(rmse_list), use_mg_dl)
    scaled_tg = np.array([(prediction_horizon - val) / prediction_horizon for val in tg_list])
    scaled_g_mean = np.array([1.0 - val for val in g_mean_list])  # lower g_mean => higher penalty
    # combine (weights can be tuned)
    score = scaled_rmse + (1.0 - scaled_tg) + scaled_g_mean
    return score


In [9]:
# Cell 7: Testing and Evaluation (plots + metrics)
# Produces: plots for three test subjects not used in training, using persistence, bergman, lstm, ml-corrector
# Also computes RMSE/MAE, Clarke & Parkes fallback, glycemia confusion matrices.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from collections import Counter

# select three test subjects (if available)
test_sids = list(per_subject_test.keys())
if len(test_sids) == 0:
    raise RuntimeError("No test subjects found. Check preprocessing cell.")
selected = test_sids[:3]

models = {
    'persistence': persistence_model['predict'],
    'bergman': bergman_model['predict'],
    'lstm': lambda s: lstm_predict_on_subject(s),
    'ml_corrector': ml_corrector_model['predict']
}

# helper to invert scaling
def inv_cgm_with_cg_scaler(y_scaled):
    if y_scaled.size == 0:
        return y_scaled
    shape = y_scaled.shape
    flat = y_scaled.reshape(-1,1)
    inv = cg_scaler.inverse_transform(flat).reshape(shape)
    return inv

# collect aggregated final-step true/pred for metrics
y_true_all = {m:[] for m in models}
y_pred_all = {m:[] for m in models}

for sid in selected:
    subj_info = per_subject_test[sid]
    X_test = subj_info['X']
    y_test = subj_info['y']  # scaled
    if X_test.shape[0] == 0:
        continue
    # get predictions from all models for this subject
    preds = {}
    for mname, mpredict in models.items():
        yhat = mpredict(sid)  # scaled predictions shape (n_samples, n_out, 1)
        preds[mname] = yhat

    # For plotting, choose one representative window (first sample)
    n_plot = min(5, X_test.shape[0])
    # We'll plot mean across first n_plot windows to smooth noise
    plt.figure(figsize=(8,4))
    tmins = np.arange(0, 5*N_OUT, 5)  # [0,5,...30-5]
    # actual: take average actual mg/dL across first n_plot windows' horizons
    actual_scaled = y_test[:n_plot].mean(axis=0).reshape(N_OUT,1)  # avg across windows
    actual = inv_cgm_with_cg_scaler(actual_scaled).reshape(-1)
    plt.plot(tmins, actual, '-o', label='Actual (avg)', linewidth=2)
    for mname, yhat_scaled in preds.items():
        if yhat_scaled.size == 0:
            continue
        yhat_avg = yhat_scaled[:n_plot].mean(axis=0).reshape(N_OUT,1)
        yhat = inv_cgm_with_cg_scaler(yhat_avg).reshape(-1)
        plt.plot(tmins, yhat, '-o', label=mname)
        # append final-step arrays for aggregate metrics
        # choose final step (index -1)
        y_true_all[mname].append(actual[-1])
        y_pred_all[mname].append(yhat[-1])
    plt.xlabel('Minutes ahead')
    plt.ylabel('Glucose (mg/dL)')
    plt.title(f'Subject {sid} prediction comparison (avg of first {n_plot} windows)')
    plt.xticks(tmins)
    plt.legend()
    plt.grid(True)
    plt.show()

# Now compute aggregated metrics across these chosen subjects (final-step only)
for mname in models:
    y_t = np.array(y_true_all[mname])
    y_p = np.array(y_pred_all[mname])
    if y_t.size == 0:
        print(f"No data for model {mname}")
        continue
    glucose_rmse = np.sqrt(mean_squared_error(y_t, y_p))
    glucose_mae  = mean_absolute_error(y_t, y_p)
    tg = temporal_gain(y_t, y_p, prediction_horizon=5*N_OUT)
    gm = g_mean(y_t, y_p)
    print(f"\nModel: {mname}")
    print(f"  Samples: {len(y_t)}")
    print(f"  RMSE: {glucose_rmse:.3f} mg/dL")
    print(f"  MAE : {glucose_mae:.3f} mg/dL")
    print(f"  Temporal gain (minutes): {tg}")
    print(f"  G-Mean (recall geometric mean): {gm:.4f}")

# Clarke/Parkes / glycemia confusion: use fallback implementations (compact)
def clarke_zone_for_point(ref, pred):
    if np.isnan(ref) or np.isnan(pred): return None
    if (ref >= 70 and abs(pred - ref) <= 0.2 * ref) or (ref < 70 and abs(pred - ref) <= 20): return 'A'
    if (pred >= 330 and ref <= 50) or (pred <= 50 and ref >= 330): return 'E'
    if ref < 70 and pred >= 180: return 'C'
    if ref >= 180 and pred < 70: return 'D'
    return 'B'

def parkes_zone_for_point(ref, pred):
    if np.isnan(ref) or np.isnan(pred): return None
    err = pred - ref; rel_err = abs(err) / max(ref,1.0)
    if (ref < 70 and abs(err) <= 15) or (ref >= 70 and rel_err <= 0.2): return 'A'
    if rel_err <= 0.35: return 'B'
    if (ref < 70 and pred > 180) or (ref > 180 and pred < 70): return 'D'
    if rel_err > 0.6: return 'E'
    return 'C'

# compute Clarke/Parkes & glycemia confusion for LSTM as example
# gather all true/pred pairs across selected subjects for LSTM
for mname in models:
    y_t = np.array(y_true_all[mname]); y_p = np.array(y_pred_all[mname])
    if y_t.size == 0: continue
    cz = [clarke_zone_for_point(r,p) for r,p in zip(y_t,y_p)]
    pz = [parkes_zone_for_point(r,p) for r,p in zip(y_t,y_p)]
    c_counts = Counter(cz); p_counts = Counter(pz)
    total = len(y_t)
    print(f"\nClarke zones for {mname}: ", {k:(c_counts[k], 100*c_counts[k]/total) for k in c_counts})
    print(f"Parkes zones for {mname}: ", {k:(p_counts[k], 100*p_counts[k]/total) for k in p_counts})

# Glycemia detection confusion matrix for LSTM (example)
def glycemia_class(x):
    x = np.asarray(x)
    cls = np.full(x.shape, -1, dtype=int)
    cls[x < 70] = 0
    cls[(x >= 70) & (x <= 180)] = 1
    cls[x > 180] = 2
    return cls

m = 'lstm'
if len(y_true_all[m])>0:
    y_t = np.array(y_true_all[m]); y_p = np.array(y_pred_all[m])
    tcls = glycemia_class(y_t); pcls = glycemia_class(y_p)
    conf = np.zeros((3,3), dtype=int)
    for t,pred in zip(tcls, pcls):
        if 0 <= t <=2 and 0 <= pred <= 2:
            conf[t,pred] += 1
    pct = conf.astype(float)
    for i in range(3):
        s = conf[i].sum()
        if s>0: pct[i] = conf[i] / float(s)
    print(f"\nGlycemia confusion matrix (rows=true, cols=pred) for {m}:")
    print(conf)
    print("Percent per row:")
    print(np.round(pct * 100,2))


248/248 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━

KeyboardInterrupt: 

## Feature Importance - SHAP

In [ ]:
# Cell 8: SHAP analysis for vanilla-LSTM and ML-corrector (final-step importance)
# Note: KernelExplainer can be slow. We use a small background sample for speed.

import shap
import numpy as np
import matplotlib.pyplot as plt

# We will explain the final-step prediction (last horizon step) from the LSTM model.
# Prepare a wrapper that maps (n_in,n_feats) input -> final-step prediction in mg/dL

def lstm_predict_final_mgdl(X):
    # X shape (m, n_in, n_feats)
    yhat_flat = lstm_model.predict(X, batch_size=64)
    yhat = yhat_flat.reshape(-1, n_out)  # scaled
    final_scaled = yhat[:, -1].reshape(-1,1)
    final_mgdl = cg_scaler.inverse_transform(final_scaled).reshape(-1)
    return final_mgdl

# Build background sample from training set (small)
bg_sample = X_tr_all[np.random.choice(max(1, X_tr_all.shape[0]), size=min(50, X_tr_all.shape[0]), replace=False)]

explainer = shap.KernelExplainer(lambda x: lstm_predict_final_mgdl(x), bg_sample)
# choose a small set of test windows to explain
# gather a few windows from test subjects (first 20 samples aggregated)
test_windows = []
for sid, info in per_subject_test.items():
    if info['X'].shape[0] > 0:
        test_windows.append(info['X'][0])
    if len(test_windows) >= 20:
        break
test_windows = np.stack(test_windows) if len(test_windows) > 0 else bg_sample[:5]

# Kernel shap expects 2D arrays; flatten time x features into one vector per sample
def reshape_for_shap(X):
    return X.reshape(X.shape[0], -1)

shap_vals = explainer.shap_values(reshape_for_shap(test_windows), nsamples=200)

# Summarize shap values across time and features: reshape back
shap_vals = np.array(shap_vals)  # list -> array
# shap for regression returns 1-d array per sample: here shap_vals shape (n_samples, n_flat)
if shap_vals.ndim == 2:
    mean_abs = np.mean(np.abs(shap_vals), axis=0)  # mean abs shap per flattened feature
    # reshape to (n_in, n_feats)
    mean_abs_ts = mean_abs.reshape(N_IN, -1)
    # sum across timesteps to get per-feature importance
    per_feature_importance = mean_abs_ts.sum(axis=0)
    feat_names = ['CGM'] + ['carbs','bolus','basal','galvanic_skin_response','skin_temp',
                            'acceleration','workout_intensity','workout_duration','heartrate','air_temp','steps']
    plt.figure(figsize=(8,4))
    plt.bar(np.arange(len(feat_names)), per_feature_importance)
    plt.xticks(np.arange(len(feat_names)), feat_names, rotation=45, ha='right')
    plt.title('LSTM (final-step) SHAP feature importance (aggregated across history)')
    plt.tight_layout()
    plt.show()

# For the ML-corrector: explain the corrector's residual prediction final step
def corr_predict_final_mgdl(X):
    # X shape (m, n_in, n_feats) -> predict resid (scaled) final then add bergman estimate to get final mg/dL
    resid_scaled = model_corr.predict(X)  # shape (m, n_out)
    resid_final = resid_scaled[:, -1].reshape(-1,1)
    # need bergman baseline for each sample to add: but KernelExplainer calls with arbitrary X not necessarily aligned to subjects.
    # We'll approximate bergman baseline by using the last CGM in the input sequence as G_init and zeros for carbs/bolus
    X_reshaped = X.reshape(X.shape[0], N_IN, n_feats)
    last_cgm_scaled = X_reshaped[:, -1, 0]  # scaled
    last_cgm = cg_scaler.inverse_transform(last_cgm_scaled.reshape(-1,1)).reshape(-1)
    # approximate bergman one-step forward as last_cgm (simple baseline) and then add residual in mg/dL
    corrected_mgdl = last_cgm + cg_scaler.inverse_transform(resid_final).reshape(-1)
    return corrected_mgdl

bg_sample_corr = bg_sample  # reuse same background
explainer_corr = shap.KernelExplainer(lambda x: corr_predict_final_mgdl(x.reshape(-1, N_IN, n_feats)), bg_sample_corr)
test_w = test_windows
shap_vals_corr = explainer_corr.shap_values(reshape_for_shap(test_w), nsamples=200)
if isinstance(shap_vals_corr, np.ndarray):
    mean_abs_corr = np.mean(np.abs(shap_vals_corr), axis=0).reshape(N_IN, -1).sum(axis=0)
    plt.figure(figsize=(8,4))
    feat_names = ['CGM'] + ['carbs','bolus','basal','galvanic_skin_response','skin_temp',
                            'acceleration','workout_intensity','workout_duration','heartrate','air_temp','steps']
    plt.bar(np.arange(len(feat_names)), mean_abs_corr)
    plt.xticks(np.arange(len(feat_names)), feat_names, rotation=45, ha='right')
    plt.title('ML-corrector SHAP feature importance (aggregated across history)')
    plt.tight_layout()
    plt.show()

print("SHAP analysis complete. Note: KernelExplainer is approximate and can be slow; for large datasets prefer model-specific explainers.")
